In [1]:
!pip install qiskit qiskit-machine-learning 

In [2]:

import matplotlib.pyplot as plt
import numpy as np
from torch import Tensor
from torch.nn import Linear, CrossEntropyLoss, MSELoss
from torch.optim import LBFGS
from matplotlib.colors import ListedColormap
from PIL import Image

In [3]:
# Load & process MNIST dataset:

from sklearn.datasets import make_classification
from datasets import load_dataset
import os

current_dir = os.getcwd()
mnist_dataset = os.path.join(".MNIST_digits", "mnist", "train-00000-of-00001.parquet")
path_to_mnist_dataset = os.path.join(current_dir, mnist_dataset)

def load_and_process_mnist_dataset(labels_to_include=[0, 1], n_samples_per_label=100, resize=(8, 8)):
	mnist_dataset = load_dataset("parquet", data_files=path_to_mnist_dataset)

	selected_images = []
	for image_idx in range(len(mnist_dataset["train"])):
		# First, select only images with labels in labels_to_include:
		
		if mnist_dataset["train"][image_idx]["label"] in labels_to_include:
			img_8x8 = mnist_dataset["train"][image_idx]["image"].resize(resize, resample=Image.BILINEAR)
			img = np.array(img_8x8)
			selected_images.append({"image": img, "label": mnist_dataset["train"][image_idx]["label"]})

	X = np.array([item["image"] for item in selected_images])
	y = np.array([item["label"] for item in selected_images])

	return X, y
	
X, y = make_classification(n_samples=200, n_features=8, n_classes=2, n_informative=8, n_redundant=0, random_state=42)

In [5]:
from qiskit.circuit.library import zz_feature_map
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
import matplotlib.pyplot as plt

def circuit_builder(vector_dim: int):
	'''Build a quantum circuit for the given feature vector using the ZZFeatureMap from Qiskit.'''
	# Encoding part:
	circuit = zz_feature_map(feature_dimension=vector_dim, reps=1)
	input_params = circuit.parameters
	
	# Ansatz part:
	ansatz = QuantumCircuit(vector_dim, name="VQC")
	thetas = ParameterVector("theta", length=3 * vector_dim)

	for i in range(vector_dim):
		ansatz.rx(thetas[i], i)
		ansatz.ry(thetas[i + vector_dim], i)
		ansatz.rz(thetas[i + 2 * vector_dim], i)

	# append entire block at once
	circuit.barrier()
	circuit = circuit.compose(ansatz)

	return circuit, input_params, thetas

circ, input_params, weigth_params = circuit_builder(vector_dim=8)

# circ.draw("mpl")


	

In [6]:
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit.primitives import StatevectorSampler as Sampler
from qiskit_machine_learning.connectors import TorchConnector
import torch
import torch.nn as nn
from torch.optim import Adam

# Map bitstrings to 2 classes (parity), otherwise SamplerQNN outputs 2**num_qubits classes.
def parity(x: int) -> int:
    return f"{x:b}".count("1") % 2

sampler_qnn_circ = SamplerQNN(
    circuit=circ,
    input_params=input_params,
    weight_params=weigth_params,
    interpret=parity,
    output_shape=2,
    sparse=False,
    sampler=Sampler(),
)

initial_weights = np.random.uniform(-0.1, 0.1, size=sampler_qnn_circ.num_weights)


class SimpleQNN(nn.Module):
    def __init__(self, qnn: SamplerQNN):
        super(SimpleQNN, self).__init__()
        self.qnn_layer = TorchConnector(qnn, initial_weights=initial_weights)

    def forward(self, x):
        return self.qnn_layer(x)


def train_QNN(qnn: SamplerQNN, X_train, y_train, num_iterations=100):
    """Train the quantum neural network."""
    qnn_in_torch = SimpleQNN(qnn)
    optimizer = Adam(qnn_in_torch.parameters(), lr=0.02)
    # SamplerQNN returns probabilities, so MSE against one-hot labels is stable.
    loss_fn = nn.MSELoss()

    # Scale features to [0, pi] for angle encoding circuits.
    mins = X_train.min(dim=0, keepdim=True).values
    maxs = X_train.max(dim=0, keepdim=True).values
    X_scaled = (X_train - mins) / (maxs - mins + 1e-8) * torch.pi

    y_onehot = torch.nn.functional.one_hot(y_train, num_classes=2).to(torch.float32)

    qnn_in_torch.train()
    for iteration in range(num_iterations):
        optimizer.zero_grad()
        outputs = qnn_in_torch(X_scaled)
        loss = loss_fn(outputs, y_onehot)
        loss.backward()
        optimizer.step()

        if (iteration + 1) % 10 == 0 or iteration == 0:
            with torch.no_grad():
                preds = torch.argmax(outputs, dim=1)
                acc = (preds == y_train).float().mean().item()
            print(f"Iteration {iteration + 1}/{num_iterations}, Loss: {loss.item():.4f}, Accuracy: {acc:.3f}")

    return qnn_in_torch

model = train_QNN(
    sampler_qnn_circ,
    torch.tensor(X, dtype=torch.float32),
    torch.tensor(y, dtype=torch.long),
    num_iterations=100,
)



Iteration 1/100, Loss: 5.5455
Iteration 2/100, Loss: 5.5455
Iteration 3/100, Loss: 5.5456
Iteration 4/100, Loss: 5.5456


KeyboardInterrupt: 